In [1]:
!pip install datasets

  Using cached datasets-4.5.0-py3-none-any.whl.metadata (19 kB)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached multiprocess-0.70.18-py313-none-any.whl.metadata (7.2 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
Using cached datasets-4.5.0-py3-none-any.whl (515 kB)
Using cached dill-0.4.0-py3-none-any.whl (119 kB)
Using cached fsspec-2025.10.0-py3-none-any.whl (200 kB)
Using cached multiprocess-0.70.18-py313-none-any.whl (151 kB)
   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   -- ------------------------------------- 1.8/27.5 MB 11.7 MB/s eta 0:00:03
   ------ --------------------------------- 4.5/27.5 MB 12.0 MB/s eta 0:00:02
   ---------- ----------------------------- 7.1/27.5 MB 12.1 MB/s eta 0:00:02
   ------------- -------------------------- 9.4/27.5 MB 12.2 MB/s eta 0:00:02
   ----------------- ---------------------- 12.1/27.5 MB 12.2 MB/s eta 0:00:02
   --------------------- ------------------ 14.7/27.5 


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Scarico il dataset delle review dei film

In [1]:
from datasets import load_dataset
dataset = load_dataset('stanfordnlp/imdb')

In [2]:
training_dataset = dataset['train']
test_dataset = dataset['test'].shuffle(seed=42).select(range(2000))

In [3]:
print(f"Righe di training: {len(training_dataset)}")
print(f"Righe di test: {len(test_dataset)}")
print("Esempio:", training_dataset[0])

Righe di training: 25000
Righe di test: 2000
Esempio: {'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic

## Preprocessiamo il testo. Creiamo il vocabolario e i dataloader

In [5]:
import torch
import torch.nn as nn
import string
from torch.utils.data import DataLoader
from datasets import load_dataset
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

# 1. Carichiamo il dataset

# 2. Creiamo il Vocabolario a partire dal tokenizzatore che semplicemente divide in parole e toglie la punteggiatura
def tokenizer(text):
    # Rimuove punteggiatura e mette minuscolo
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.lower().split()


#Contiamo il numero di parole per selezionare solo le più frequenti
counter = Counter()
for text in dataset['train']['text']:
    counter.update(tokenizer(text.lower()))

# Teniamo solo le 25.000 parole più frequenti e creiamo il vocabolario
vocab = {word: i+2 for i, (word, _) in enumerate(counter.most_common(25000))}
vocab['<PAD>'] = 0  # Padding (riempitivo)
vocab['<UNK>'] = 1  # Unknown (parole sconosciute)

import json
with open('model/vocab.json', 'w') as f:
    json.dump(vocab, f)

# Funzione per convertire testo in indici
def text_pipeline(text):
    #Se non trova il token lo sostituisce con '<UNK>'
    return [vocab.get(token, 1) for token in tokenizer(text.lower())]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. Collate Function (Per gestire batch con frasi di lunghezza diversa)

#la funzione di collate è una funzione di trasformazione del batch
def collate_batch(batch):
    label_list, text_list = [], []
    for _item in batch:
        # Trasformiamo il testo in numeri
        processed_text = torch.tensor(text_pipeline(_item['text']), dtype=torch.int64)
        text_list.append(processed_text)
        label_list.append(_item['label'])

    label_list = torch.tensor(label_list, dtype=torch.float32).to(device)
    # pad_sequence aggiunge zeri alle frasi più corte per pareggiare la lunghezza. Lo zero corriponde all'indice del token <PAD>
    text_list = pad_sequence(text_list, batch_first=True, padding_value=0).to(device)
    return text_list, label_list

# Creiamo i DataLoader
train_loader = DataLoader(dataset['train'], batch_size=64, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(dataset['test'], batch_size=64, shuffle=False, collate_fn=collate_batch)

## Definiamo ora la rete RNN

In [6]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, dropout_rate=0.1):
        super(RNNClassifier, self).__init__()

        # 1. Embedding: Trasforma l'indice (es. 45) in un vettore denso
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # 2. LSTM Layer
        # batch_first=True significa che l'input è (batch, seq_len, features)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, bidirectional=True,
                            batch_first=True)

        # 3. Fully Connected Layer (Output). Devo moltiplicare l'hidden_dim per 2 perchè la rete è bidirezionale
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

        #Il dropout serve a regolarizzare la rete. Praticamente spegne il dropout_rate per cento dei neuroni ad ogni iterazione
        self.dropout = nn.Dropout(dropout_rate)
        # 4. Sigmoide (per ottenere probabilità tra 0 e 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, text):
        # text shape: [batch_size, seq_len]
        embedded = self.embedding(text)

        # rnn output: [batch_size, seq_len, hidden_dim]
        #hidden ha shape: [num_layers * num_directions, batch_size, hidden_dim]
        output, (hidden, cell) = self.rnn(embedded)
        hidden_forward = hidden[-2,:,:]
        hidden_backward = hidden[-1,:,]
        hidden_final = torch.cat((hidden_forward, hidden_backward), dim=1)
        # shape risultante: [batch_size, hidden_dim * 2]
        hidden_final = self.dropout(hidden_final)
        return self.sigmoid(self.fc(hidden_final))

# Iperparametri
VOCAB_SIZE = len(vocab)
EMBED_DIM = 64
HIDDEN_DIM = 64
OUTPUT_DIM = 1 # 1 perché è classificazione binaria (0 o 1)

model = RNNClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)
print(model)

RNNClassifier(
  (embedding): Embedding(25002, 64, padding_idx=0)
  (rnn): LSTM(64, 64, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (sigmoid): Sigmoid()
)


## Creo il loop di training

In [7]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

def train_one_epoch(model, iterator, optimizer, criterion):
    model.train()
    epoch_loss = 0
    epoch_acc = 0

    for text, labels in iterator:
        optimizer.zero_grad()

        # Forward pass
        # output shape: [64, 1]
        # labels shape: [64]
        # Devo fare lo squueze sull'output per trasformarlo da matrice a vettore!
        predictions = model(text).squeeze(1)

        # Calcolo loss e accuracy
        loss = criterion(predictions, labels)
        rounded_preds = torch.round(predictions)
        correct = (rounded_preds == labels).float()
        acc = correct.sum() / len(correct)

        # Backward pass
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)



## Creo la funzione di valutazione dell'accuratezza

In [8]:
def evaluate(model, iterator, criterion):
    # Mette il modello in modalità "valutazione"
    # (spegne Dropout )
    model.eval()

    epoch_loss = 0
    epoch_acc = 0

    # Disabilita il calcolo dei gradienti (velocizza e risparmia RAM)
    with torch.no_grad():

        for text, labels in iterator:
            # Forward pass
            predictions = model(text).squeeze(1)

            # Calcolo loss
            loss = criterion(predictions, labels)

            # Calcolo accuracy
            rounded_preds = torch.round(predictions)
            correct = (rounded_preds == labels).float()
            acc = correct.sum() / len(correct)

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

## Imposto il training loop

In [10]:
import time

N_EPOCHS = 5

for epoch in range(N_EPOCHS):

    start_time = time.time()

    # 1. Addestramento
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)

    # 2. Valutazione
    test_loss, test_acc = evaluate(model, test_loader, criterion)

    end_time = time.time()
    mins, secs = divmod(end_time - start_time, 60)

    print(f'Epoch: {epoch+1:02} | Time: {int(mins)}m {int(secs)}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\tTest  Loss: {test_loss:.3f} | Test  Acc: {test_acc*100:.2f}%')

Epoch: 01 | Time: 0m 10s
	Train Loss: 0.413 | Train Acc: 82.33%
	Test  Loss: 0.397 | Test  Acc: 83.65%
Epoch: 02 | Time: 0m 10s
	Train Loss: 0.263 | Train Acc: 90.40%
	Test  Loss: 0.376 | Test  Acc: 84.80%
Epoch: 03 | Time: 0m 10s
	Train Loss: 0.208 | Train Acc: 92.95%
	Test  Loss: 0.369 | Test  Acc: 85.79%
Epoch: 04 | Time: 0m 10s
	Train Loss: 0.166 | Train Acc: 94.67%
	Test  Loss: 0.383 | Test  Acc: 86.29%
Epoch: 05 | Time: 0m 10s
	Train Loss: 0.132 | Train Acc: 96.10%
	Test  Loss: 0.420 | Test  Acc: 86.22%


In [11]:
def predict_sentiment(model, sentence, vocab, device):
    model.eval() # Importante: spegne il Dropout per avere risultati stabili

    # 1. Tokenizzazione (uguale a quella fatta nel training)
    tokens = sentence.lower().split()

    # 2. Convertiamo parole in numeri
    # Se la parola non è nel vocabolario, usiamo l'indice 1 (<UNK>)
    indexed = [vocab.get(t, 1) for t in tokens]


    # 3. Trasformiamo in Tensore PyTorch
    tensor = torch.LongTensor(indexed).to(device)

    # 4. Aggiungiamo la dimensione del Batch
    # Il modello si aspetta [batch_size, seq_len], noi abbiamo solo [seq_len].
    # unsqueeze(0) trasforma [10] in [1, 10]
    tensor = tensor.unsqueeze(0)

    # 5. Predizione
    with torch.no_grad():
        prediction = model(tensor)

    # Restituiamo il valore float (da 0 a 1)
    return prediction.item()

In [12]:
# Definiamo alcune frasi di test
test_sentences = [
    "This movie is absolutely fantastic and wonderful",  # Chiaramente Positiva
    "The plot was boring and the acting was terrible",   # Chiaramente Negativa
    "It was not bad, actually quite good",               # Complessa (doppia negazione/contrasto)
    "I wasted two hours of my life",                     # Negativa indiretta
    "A masterpiece"                                      # Molto corta
]

print("-" * 50)
print(f"{'FRASE':<50} | {'SCORE':<10} | {'SENTIMENT'}")
print("-" * 50)

for sentence in test_sentences:
    score = predict_sentiment(model, sentence, vocab, device)

    # Interpretazione semplice dello score
    if score >= 0.5:
        label = "POSITIVO"
    else:
        label = "NEGATIVO "

    print(f"{sentence[:47]:<50}... | {score:.4f}     | {label}")

--------------------------------------------------
FRASE                                              | SCORE      | SENTIMENT
--------------------------------------------------
This movie is absolutely fantastic and wonderfu   ... | 0.9970     | POSITIVO
The plot was boring and the acting was terrible   ... | 0.0324     | NEGATIVO 
It was not bad, actually quite good               ... | 0.5375     | POSITIVO
I wasted two hours of my life                     ... | 0.4490     | NEGATIVO 
A masterpiece                                     ... | 0.9733     | POSITIVO


It was not bad, actually quite good               ... | 0.3572     | NEGATIVO

Qui il modello è abbastanza confuso e sbaglia. Stiamo re-imparando non solo la semantica ma anche la sintassi dell'inglese. Possiamo ora al posto di allenare la matrice degli embedding usare glove, dove già le parole hanno una loro rappresentazione semantica.

In [17]:
!pip install torchtext

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 9.0 MB/s eta 0:00:00

   ---------------------------------------- 2/2 [torchtext]




[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import torchtext

# 1. Scarichiamo i vettori GloVe (versione 6B, 100 dimensioni)
print("Scaricando GloVe...")
glove = torchtext.vocab.GloVe(name='6B', dim=100)

# 2. Creiamo la matrice dei pesi
embedding_matrix = torch.zeros((len(vocab), 100))

print("Creazione matrice dei pesi...")
hit = 0
miss = 0

for word, index in vocab.items():
    # Se la parola è nel vocabolario di GloVe, prendiamo il suo vettore
    if word in glove.stoi:
        embedding_matrix[index] = glove[word]
        hit += 1
    else:
        # Se non c'è (es. <PAD>, <UNK> o nomi strani), lasciamo zeri o inizializziamo random
        # Qui lasciamo random normal per <UNK>)
        if word == '<UNK>':
             embedding_matrix[index] = torch.randn(100)
        miss += 1

print(f"Parole trovate in GloVe: {hit}, Non trovate: {miss}")

Scaricando GloVe...
Creazione matrice dei pesi...
Parole trovate in GloVe: 22987, Non trovate: 2015


In [16]:
# Aggiorniamo la dimensione dell'embedding a 100
EMBED_DIM = 100
HIDDEN_DIM = 32
# Ricreiamo il modello con la nuova dimensione
model = RNNClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM, dropout_rate=0.5).to(device)

# Copiamo i pesi di GloVe dentro il layer di embedding del modello
model.embedding.weight.data.copy_(embedding_matrix)

# Gestione <PAD>: Il padding deve rimanere sempre zero, non deve imparare nulla
model.embedding.weight.data[vocab['<PAD>']] = torch.zeros(EMBED_DIM)

print("Pesi GloVe caricati con successo!")

Pesi GloVe caricati con successo!


Aggiungiamo il learning decay e il salvataggio del miglior modello ad ogni iterazione

In [17]:
import time
from torch.optim.lr_scheduler import ReduceLROnPlateau
N_EPOCHS = 10
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
best_valid_loss = float('inf')
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.66, patience=2)
for epoch in range(N_EPOCHS):

    start_time = time.time()

    # 1. Addestramento
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)

    # 2. Valutazione
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    scheduler.step(test_loss)

    if test_loss < best_valid_loss:
        best_valid_loss = test_loss
        # Salviamo i pesi su disco
        torch.save(model.state_dict(), './model/miglior_modello_imdb.pt')
        print(f'Epoch: {epoch+1:02} | Nuovo Record! Modello salvato (Loss: {test_loss:.3f})')
    else:
        print(f'Epoch: {epoch+1:02} | Nessun miglioramento.')
    end_time = time.time()
    mins, secs = divmod(end_time - start_time, 60)

    print(f'Epoch: {epoch+1:02} | Time: {int(mins)}m {int(secs)}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\tTest  Loss: {test_loss:.3f} | Test  Acc: {test_acc*100:.2f}%')

Epoch: 01 | Nuovo Record! Modello salvato (Loss: 0.578)
Epoch: 01 | Time: 0m 9s
	Train Loss: 0.664 | Train Acc: 59.45%
	Test  Loss: 0.578 | Test  Acc: 71.35%
Epoch: 02 | Nuovo Record! Modello salvato (Loss: 0.386)
Epoch: 02 | Time: 0m 9s
	Train Loss: 0.470 | Train Acc: 79.79%
	Test  Loss: 0.386 | Test  Acc: 84.25%
Epoch: 03 | Nuovo Record! Modello salvato (Loss: 0.344)
Epoch: 03 | Time: 0m 9s
	Train Loss: 0.281 | Train Acc: 90.24%
	Test  Loss: 0.344 | Test  Acc: 86.46%
Epoch: 04 | Nessun miglioramento.
Epoch: 04 | Time: 0m 9s
	Train Loss: 0.185 | Train Acc: 94.27%
	Test  Loss: 0.364 | Test  Acc: 87.22%
Epoch: 05 | Nessun miglioramento.
Epoch: 05 | Time: 0m 9s
	Train Loss: 0.125 | Train Acc: 96.74%
	Test  Loss: 0.434 | Test  Acc: 86.12%
Epoch: 06 | Nessun miglioramento.
Epoch: 06 | Time: 0m 9s
	Train Loss: 0.088 | Train Acc: 97.96%
	Test  Loss: 0.427 | Test  Acc: 85.91%
Epoch: 07 | Nessun miglioramento.
Epoch: 07 | Time: 0m 8s
	Train Loss: 0.061 | Train Acc: 98.73%
	Test  Loss: 0.516 | 

KeyboardInterrupt: 

In [18]:
# 1. Ricrea l'architettura
model = RNNClassifier(len(vocab), 100, HIDDEN_DIM, OUTPUT_DIM, 0.1).to(device)

# 2. Carica i pesi dal file
model.load_state_dict(torch.load('./model/miglior_modello_imdb.pt'))
model.eval() # Metti in eval mode

print("Modello caricato con successo!")

# 3. Riprova la frase trappola
frase = "It was not bad, actually quite good"
score = predict_sentiment(model, frase, vocab, device)
print(f"Score finale: {score:.4f}")

Modello caricato con successo!
Score finale: 0.4471
